# Pancake Problem: Baseline Submission

This notebook demonstrates a simple pancake-sort baseline that generates a Kaggle submission.
Moves use the prefix-reversal notation `Rk`, where `Rk` flips the first `k` elements of a state.

The work was carried out in a Docker image, so some commands for working with files may be unclear.

In [ ]:
from __future__ import annotations

import pandas as pd
import numpy as np

from pathlib import Path
from typing import Iterable

from tqdm.notebook import tqdm

import gc

In [ ]:
# Detect Kaggle environment ( /kaggle/input/... ), otherwise fall back to local repo layout.
DATA_ROOT = Path('/content/Pancake')
TEMP_ROOT = Path('/content/Pancake/Temp')
assert DATA_ROOT.exists(), f'Dataset directory not found: {DATA_ROOT!s}'

TEST_PATH = DATA_ROOT / 'test.csv'
SUBMISSION_PATH = DATA_ROOT / 'submission.csv' # Kaggle expects this in the working directory
BEST_SUBMISSION_PATH = DATA_ROOT / 'collect_best.csv'

In [ ]:
test_df = pd.read_csv(TEST_PATH)
best_df = pd.read_csv(BEST_SUBMISSION_PATH)

## def

In [ ]:
def parse_permutation(raw: str) -> list[int]:
    """Parse a comma-separated permutation string into integer positions."""
    return [int(token) for token in raw.split(',') if token]

In [ ]:
def pancake_sort_path(perm: Iterable[int]) -> list[str]:
    """Return a sequence of prefix reversals that sorts `perm` to the identity permutation."""
    arr = list(perm)
    n = len(arr)
    moves: list[str] = []

    for target in range(n, 1, -1):
        desired_value = target - 1
        idx = arr.index(desired_value)

        if idx == target - 1:
            continue  # already in place

        if idx != 0:
            moves.append(f'R{idx + 1}')
            arr[: idx + 1] = reversed(arr[: idx + 1])

        moves.append(f'R{target}')
        arr[:target] = reversed(arr[:target])

    return moves

In [ ]:
def pancake_sort_input(perm: Iterable[int]) -> list[int]:
    """Return a sequence of prefix reversals that sorts `perm` to the identity permutation."""
    arr = list(perm)
    n = len(arr)
    moves: list[int] = []

    print(f'Len: {n}, Permutation: {arr}')
    move = int(input(f'Move:{len(moves) + 1:2d} | R:').strip())

    while move:
        moves.append(move)
        arr[: move] = reversed(arr[: move])
        print(arr)
        move = int(input(f'Move:{len(moves) + 1:2d} | R:'))

    return moves

In [ ]:
def prob_step(perm):
    perm_ = perm.copy()
    n = len(perm_)
    k = perm_[-1] == n - 1
    # k -= sum(perm_[:2]) == 1

    while len(perm_) - 1:
        m = perm_.pop()
        k += perm_[-1] - 1 == m or perm_[-1] + 1 == m
    return n - k

In [ ]:
def compare():
    n_list = [5, 12, 15, 16, 20, 25, 30, 35, 40, 45, 50, 75, 100]
    for n in n_list:
        pos = (best_df.n == n)
        score = best_df.score[pos].sum()
        n_sum = best_df.n[pos].sum()
        prob_step = best_df.prob_step[pos].sum()

        print(f'n: {n} | sum n: {n_sum} | score: {score} | prob step: {prob_step} | potential: {score - prob_step}')
    print()
    print(f'sum n: {best_df.n.sum()} | score: {best_df.score.sum()} | prob step: {best_df.prob_step.sum()}')

In [ ]:
def best_solution(submission_df, best_df=None, safe=False):

    if best_df is None:
        best_df = pd.read_csv(BEST_SUBMISSION_PATH)

    best_df = best_df.set_index('id')
    submission_df = submission_df.set_index('id')

    common_idx = best_df.index.intersection(submission_df.index)

    same_score_idx = []
    best_score_idx = []

    for idx in common_idx:
        if best_df.loc[idx, 'score'] == submission_df.loc[idx, 'score']:
            same_score_idx.append(idx)

        if best_df.loc[idx, 'score'] > submission_df.loc[idx, 'score']:
            best_score_idx.append(idx)
            best_df.loc[idx, ['solution', 'score']] = submission_df.loc[idx, ['solution', 'score']]

    if safe and best_score_idx:
        best_df.to_csv(BEST_SUBMISSION_PATH, index=False)
        print('Best submission updated.')

    return best_df.reset_index().sort_values('id'), {'best': len(best_score_idx),
                                                     'same': len(same_score_idx),
                                                     'worse': len(common_idx) - len(best_score_idx) - len(same_score_idx)}


In [ ]:
def process_row(row, func=None, treshold=3, save=False, from_target=False):

    perm = parse_permutation(row['permutation'])

    if from_target:
        perm, _= revers_perm(perm)

    moves, _, mlen, i = func(perm, treshold)

    if from_target:
        steps = moves[0][::-1]
    else:
        steps = moves[0]

    path_str = '.'.join(f'R{k}' for k in steps) if moves else 'UNSOLVED'

    if save:
        id_ = row['id']
        n = int(row['n'])
        print(f'id: {id_} - complete')

        with open(TEMP_ROOT / f'n_{n}.txt', mode='a') as file_:
            file_.write(f'{id_} - ' + path_str + '\n')

    return {
        'id': row['id'],
        'permutation': row['permutation'],
        'solution': path_str,
        'score': len(steps),
        'n': int(row['n']),
        'mlen': mlen,
        'iter': i
    }

In [ ]:
def revers_perm(perm):
    code_dict = {}
    decode_dict = {}
    for i, n in enumerate(perm):
        code_dict[n] = i
        decode_dict[i] = n
    reversed_perm = [code_dict[i] for i in range(len(perm))]
    return reversed_perm, decode_dict


In [ ]:
def check_steps(df):
    for row in df.to_dict('records'):
        perm = parse_permutation(row['permutation'])
        steps = list(map(int, row['solution'].lstrip('R').split('.R')))
        for idx in steps:
            perm[:idx] = perm[:idx][::-1]
        if perm != list(range(len(perm))):
            id_perm = row['id']
            print(f'Wrong solution for id{id_perm}')

check_steps(best_df)

In [ ]:
def print_search(perm_dict):
    if perm_dict:
        for key, item in perm_dict.items():
            print(f'{key} | {item}')
    else:
        print(None)

## Old versions


In [ ]:
def pancake_sort_v2_2_np(perm):
    """Return a sequence of prefix reversals that sorts `perm` to the identity permutation."""
    arr = np.array(perm, dtype=np.uint8)
    n = len(arr)

    arr_max_len = 0
    div = 0
    total_iter = 0

    permute_search = {arr.tobytes():([], 0)}

    target = np.arange(n, dtype=np.uint8).tobytes()

    def check_and_wrigth(idx, _div=0):
        if not moves or (moves and moves[-1] != idx):
            arr_ = np.concat([arr[idx-1:: -1], arr[idx:]])
            if arr_.tobytes() not in permute_search:
                permute_search[arr_.tobytes()] = [moves + [idx], stat + _div]

    while target not in permute_search:

        stage_permute_search = permute_search.copy()

        for arr_b in stage_permute_search:

            total_iter += 1

            moves = permute_search[arr_b][0]
            stat = permute_search[arr_b][1]

            arr = np.frombuffer(arr_b, dtype=np.uint8)

            left_value = arr[0] - 1 if arr[0] > 0 else None
            rigth_value = arr[0] + 1 if arr[0] < n - 1 else None

            left_idx = np.where(arr==left_value)[0][0] if left_value is not None else 1
            rigth_idx = np.where(arr==rigth_value)[0][0] if rigth_value else n

            if left_idx != 1 and arr[left_idx - 1] + 1 == left_value:
                div = 1
            else:
                div = 0

            check_and_wrigth(left_idx, div)


            if rigth_idx != 1 and rigth_value and arr[rigth_idx - 1] == rigth_value + 1:
                div = 1
            else:
                div = 0

            check_and_wrigth(rigth_idx, div)

            if arr[0] == arr[1] + 1:
                for i in range(2, n):
                    if arr[i - 1] != arr[i] + 1:
                        check_and_wrigth(i)
                        break

            elif arr[0] + 1 == arr[1]:
                for i in range(2, n):
                    if arr[i - 1] + 1 != arr[i]:
                        check_and_wrigth(i)
                        break

            arr_max_len = max(arr_max_len, len(permute_search))
            permute_search.pop(arr_b)

    return permute_search[target], permute_search, arr_max_len, total_iter

In [ ]:
def pancake_sort_v2_3(perm, treshold=9):
    """Return a sequence of prefix reversals that sorts `perm` to the identity permutation."""
    arr = list(perm)
    n = len(arr)

    arr_max_len = 0
    div = 0
    total_iter = 0
    moves = ()

    permute_search = {tuple(arr):((), 0)}

    target = tuple(i for i in range(n))

    def check_and_write(idx, _div=0):
        _stat = stat + _div
        if (not moves or (moves and moves[-1] != idx)) and min_stat + treshold >= _stat:
            arr_ = tuple(list_arr[idx-1::-1] + list_arr[idx:])
            if arr_ not in permute_search or (len(permute_search[arr_][0]) > len(moves) and permute_search[arr_][1] > _stat):
                permute_search[arr_] = (moves + (idx,), _stat)

    while target not in permute_search:

        stage_permute_search = permute_search.copy()

        min_stat = min(i[1] for i in permute_search.values())


        for arr in stage_permute_search:

            total_iter += 1

            moves = permute_search[arr][0]
            stat = permute_search[arr][1]

            list_arr = list(arr)

            left_value = list_arr[0] - 1 if list_arr[0] > 0 else 1
            rigth_value = list_arr[0] + 1 if list_arr[0] < n - 1 else None

            left_idx = list_arr.index(left_value)
            rigth_idx = list_arr.index(rigth_value) if rigth_value else n

            if left_idx != 1 and list_arr[left_idx - 1] + 1 == left_value:
                div = 2
            else:
                div = 1

            check_and_write(left_idx, div)


            if rigth_idx != 1 and list_arr[rigth_idx - 1] - 1 == rigth_value:
                div = 2
            else:
                div = 1

            check_and_write(rigth_idx, div)

            if list_arr[0] - 1 == list_arr[1]:
                for i in range(2, n):
                    if list_arr[i - 1] - 1 != list_arr[i]:
                        check_and_write(i, 2)
                        break

            elif list_arr[0] + 1 == list_arr[1]:
                for i in range(2, n):
                    if list_arr[i - 1] + 1 != list_arr[i]:
                        check_and_write(i, 2)
                        break

            arr_max_len = max(arr_max_len, len(permute_search))
            permute_search.pop(arr)

    return permute_search[target], permute_search, arr_max_len, total_iter

In [ ]:
def pancake_sort_v3_2(perm, treshold=3):
    """Return a sequence of prefix reversals that sorts `perm` to the identity permutation."""
    arr = list(perm)
    n = len(arr)

    arr_max_len = 0
    total_iter = 0

    # permute_search = {tuple(arr):((), 0)}
    permute_search = {tuple(arr):((), 0), tuple(reversed(arr)):((n,), 0)}

    target = tuple(i for i in range(n))

    def check_and_write(idx, _div=0):
        _stat = stat + _div
        if (not moves or (moves and moves[-1] != idx)) and min_stat + treshold >= _stat:
            arr_ = tuple(list_arr[idx-1::-1] + list_arr[idx:])
            if arr_ not in permute_search or (len(permute_search[arr_][0]) > len(moves) and permute_search[arr_][1] > _stat):
                permute_search[arr_] = (moves + (idx,), _stat)

    while permute_search and target not in permute_search:

        min_stat = min(i[1] for i in permute_search.values())

        for arr in list(permute_search):

            total_iter += 1

            moves = permute_search[arr][0]
            stat = permute_search[arr][1]

            list_arr = list(arr)

            # value < arr[0] : left
            left_value = list_arr[0] - 1 if list_arr[0] else None
            left_idx = list_arr.index(left_value) if left_value is not None else None

            # arr[0] < value : rigth
            rigth_value = list_arr[0] + 1 if list_arr[0] < n - 1 else None
            rigth_idx = list_arr.index(rigth_value) if rigth_value else n

            if rigth_idx == 1:
                if list_arr[0] == 0:
                    for i in range(1, n):
                        if list_arr[-i] != n - i:
                            check_and_write(n - i + 1, 2)
                            break
                else:
                    for i in range(2, n):
                        if list_arr[i - 1] + 1 != list_arr[i]:
                            check_and_write(i, 2)
                            break

            elif not rigth_value or list_arr[rigth_idx - 1] - 1 != rigth_value:
                check_and_write(rigth_idx, 0)
            elif rigth_value:
                check_and_write(rigth_idx, 2)

            if left_idx and left_idx == 1:
                for i in range(2, n):
                    if list_arr[i - 1] - 1 != list_arr[i]:
                        check_and_write(i, 2)
                        break
            elif left_idx and list_arr[left_idx - 1] + 1 != left_value:
                check_and_write(left_idx, 0)
            elif left_idx:
                check_and_write(left_idx, 2)


            arr_max_len = max(arr_max_len, len(permute_search))
            permute_search.pop(arr)

        answer = permute_search.get(target)

    return answer, permute_search, arr_max_len, total_iter

In [ ]:
def pancake_sort_v3_3(perm, treshold=3):
    """Return a sequence of prefix reversals that sorts `perm` to the identity permutation."""
    arr = list(perm)
    n = len(arr)

    arr_max_len = 0
    total_iter = 0

    # permute_search = {tuple(arr):((), 0), tuple(reversed(arr)):((n,), 0)}
    permute_search = {tuple(arr):((), 0)}

    target = tuple(i for i in range(n))

    def check_and_write(idx, _div=0):
        _stat = stat + _div
        if (not moves or (moves and moves[-1] != idx)) and min_stat + treshold >= _stat:
            arr_ = tuple(list_arr[idx-1::-1] + list_arr[idx:])
            if arr_ not in permute_search or (len(permute_search[arr_][0]) > len(moves) and permute_search[arr_][1] > _stat):
                permute_search[arr_] = (moves + (idx,), _stat)

    def mask_step(perm):
            perm_ = perm.copy()
            n = len(perm_)
            step = [] if perm_[n-1] == n-1 else [n]

            while len(perm_) - 2:
                m = perm_.pop()
                if not (perm_[-1] - 1 == m or perm_[-1] + 1 == m):
                    step.append(len(perm_))
            return step


    while target not in permute_search:

        # stage_permute_search = permute_search.copy()

        min_stat = min(i[1] for i in permute_search.values())

        for arr in list(permute_search):

            total_iter += 1

            moves = permute_search[arr][0]
            stat = permute_search[arr][1]

            list_arr = list(arr)

            rigth_step = False
            left_step = False

            # value < arr[0] : left
            left_value = list_arr[0] - 1 if list_arr[0] else None
            left_idx = list_arr.index(left_value) if left_value is not None else None

            # arr[0] < value : rigth
            rigth_value = list_arr[0] + 1 if list_arr[0] < n - 1 else None
            rigth_idx = list_arr.index(rigth_value) if rigth_value else n

            if not rigth_value or (rigth_idx != 1 and list_arr[rigth_idx - 1] - 1 != rigth_value):
                check_and_write(rigth_idx, 0)
                rigth_step = True

            if left_idx and left_idx != 1 and list_arr[left_idx - 1] + 1 != left_value:
                check_and_write(left_idx, 0.2)
                left_step = True

            if not rigth_step or not left_step:
                for i in mask_step(list_arr):
                    check_and_write(i, 2)

            arr_max_len = max(arr_max_len, len(permute_search))
            permute_search.pop(arr)

    return permute_search[target], permute_search, arr_max_len, total_iter

In [ ]:
def pancake_sort_v3_5_np(perm, treshold=10):
    """Return a sequence of prefix reversals that sorts `perm` to the identity permutation."""
    arr = np.array(perm, dtype=np.uint8)
    n = len(arr)

    arr_max_len = 0
    total_iter = 0

    permute_search = {arr.tobytes(): (np.array([], dtype=np.uint8), 0)}

    target = np.arange(n, dtype=np.uint8).tobytes()

    def check_and_write(idx, _div=0):
        _stat = stat + _div
        if (not moves.size or (moves.size and moves[-1] != idx)) and min_stat + treshold >= _stat:
            arr_ = np.concatenate([list_arr[idx-1::-1], list_arr[idx:]])
            arr_ = arr_.tobytes()
            old = permute_search.get(arr_)
            if not old:
                permute_search[arr_] = (np.concatenate([moves, np.array([idx], dtype=np.uint8)], axis=0), _stat)
            else:
                old_moves, old_stat = old
                if len(old_moves) > len(moves) and old_stat > _stat:
                    permute_search[arr_] = (moves + (idx,), _stat)

    def mask_step(perm):
            perm_ = perm.copy()
            n = len(perm_)
            step = [] if perm_[n-1] == n-1 else [n]

            while len(perm_) - 2:
                m = perm_[-1:]
                perm_ = perm_[:-1]
                if not (perm_[-1] - 1 == m or perm_[-1] + 1 == m):
                    step.append(len(perm_))
            return step


    while target not in permute_search:

        min_stat = min(i[1] for i in permute_search.values())

        for arr in list(permute_search):

            total_iter += 1

            moves = permute_search[arr][0]
            stat = permute_search[arr][1]

            list_arr = np.frombuffer(arr, dtype=np.int8)

            rigth_step = False
            left_step = False

            # value < arr[0] : left
            left_value = list_arr[0] - 1 if list_arr[0] else None
            left_idx = np.where(list_arr==left_value)[0][0] if left_value is not None else None

            # arr[0] < value : rigth
            rigth_value = list_arr[0] + 1 if list_arr[0] < n - 1 else None
            rigth_idx = np.where(list_arr==rigth_value)[0][0] if rigth_value else n

            if not rigth_value or (rigth_idx != 1 and list_arr[rigth_idx - 1] - 1 != rigth_value):
                check_and_write(rigth_idx, 10 / n)
                rigth_step = True
            elif rigth_idx != 1:
                check_and_write(rigth_idx, 10)

            if left_idx and left_idx != 1 and list_arr[left_idx - 1] + 1 != left_value:
                check_and_write(left_idx, 10 / n)
                left_step = True
            elif left_idx and left_idx != 1:
                check_and_write(left_idx, 10)

            if not rigth_step or not left_step:
                for i in mask_step(list_arr):
                    check_and_write(i, 10)

            arr_max_len = max(arr_max_len, len(permute_search))
            permute_search.pop(arr)

    return permute_search[target], None, arr_max_len, total_iter

In [ ]:
def pancake_sort_v3_5(perm, treshold=3):
    """Return a sequence of prefix reversals that sorts `perm` to the identity permutation."""
    arr = list(perm)
    n = len(perm)

    arr_max_len = 0
    total_iter = 0

    # permute_search = {tuple(arr):((), 0), tuple(reversed(arr)):((n,), 0)}
    permute_search = {tuple(perm):((), 0)}

    target = tuple(i for i in range(n))

    def check_and_write(idx, _div=0):
        _stat = stat + _div
        if (not moves or (moves and moves[-1] != idx)) and min_stat + treshold >= _stat:
            arr_ = tuple(list_arr[idx-1::-1] + list_arr[idx:])
            if arr_ not in permute_search or (len(permute_search[arr_][0]) > len(moves) and permute_search[arr_][1] > _stat):
                permute_search[arr_] = (moves + (idx,), _stat)

    def mask_step(perm):
            perm_ = perm.copy()
            n = len(perm_)
            step = [] if perm_[n-1] == n-1 else [n]

            while len(perm_) - 2:
                m = perm_.pop()
                if not (perm_[-1] - 1 == m or perm_[-1] + 1 == m):
                    step.append(len(perm_))
            return step


    while target not in permute_search:

        min_stat = min(i[1] for i in permute_search.values())

        for arr in list(permute_search):

            total_iter += 1

            moves = permute_search[arr][0]
            stat = permute_search[arr][1]

            list_arr = list(arr)

            rigth_step = False
            left_step = False

            # value < arr[0] : left
            left_value = list_arr[0] - 1 if list_arr[0] else None
            left_idx = list_arr.index(left_value) if left_value is not None else None

            # arr[0] < value : rigth
            rigth_value = list_arr[0] + 1 if list_arr[0] < n - 1 else None
            rigth_idx = list_arr.index(rigth_value) if rigth_value else n

            if not rigth_value or (rigth_idx != 1 and list_arr[rigth_idx - 1] - 1 != rigth_value):
                check_and_write(rigth_idx, 10 / n)
                rigth_step = True
            elif rigth_idx != 1:
                check_and_write(rigth_idx, 10)

            if left_idx and left_idx != 1 and list_arr[left_idx - 1] + 1 != left_value:
                check_and_write(left_idx, 10 / n)
                left_step = True
            elif left_idx and left_idx != 1:
                check_and_write(left_idx, 10)

            if not rigth_step or not left_step:
                for i in mask_step(list_arr):
                    check_and_write(i, 10)

            arr_max_len = max(arr_max_len, len(permute_search))
            permute_search.pop(arr)

    return permute_search[target], permute_search, arr_max_len, total_iter

In [ ]:
def pancake_sort_recursive_v2_1(perm, threshold=0):
    n = len(perm)

    min_moves = n + 2
    permute_complete = {}

    target = tuple(i for i in range(n))

    # ---------------------------------------------------------------------------------

    def idxs_and_positions(arr):

        # value < arr[0] : left
        left_value = arr[0] - 1 if arr[0] else None
        left_idx = arr.index(left_value) if left_value is not None else None

        # arr[0] < value : right
        right_value = arr[0] + 1 if arr[0] < n - 1 else None
        right_idx = arr.index(right_value) if right_value else n

        return left_value, left_idx, right_value, right_idx


    def mask_step_add(perm):
        perm_ = list(perm)
        n = len(perm_)
        steps = [] if perm_[n-1] == n-1 else [n]

        _, left_idx, _, right_idx = idxs_and_positions(perm)

        while len(perm_) - 2:
            m = perm_.pop()
            if not (perm_[-1] - 1 == m or perm_[-1] + 1 == m):
                steps.append(len(perm_))

        if left_idx and left_idx not in steps:
            steps.append(left_idx)

        if right_idx and right_idx not in steps:
            steps.append(right_idx)

        return steps

    def mask_step(perm):
        perm_ = list(perm)
        n = len(perm_)
        step = [] if perm_[n - 1] == n - 1 else [n]

        while len(perm_) - 2:
            m = perm_.pop()
            if not (perm_[-1] - 1 == m or perm_[-1] + 1 == m):
                step.append(len(perm_))
        return step


    def move_recursive(arr, moves, div=0):

        len_moves = len(moves)
        if min_moves < len_moves:
            return

        def check_and_write(idx, _div=0):
            nonlocal min_moves
            if (not moves or (moves and moves[-1] != idx)):
                arr_ = arr[idx - 1::-1] + arr[idx:]

                if arr_ == target:
                    permute_complete[moves + (idx,)] = len_moves + 1
                    min_moves = min(min_moves, len_moves)
                else:
                    move_recursive(arr_, moves + (idx,), _div)


        left_value, left_idx, right_value, right_idx = idxs_and_positions(arr)

        right_step = False
        left_step = False


        if not right_value or (right_idx != 1 and arr[right_idx - 1] - 1 != right_value):
            check_and_write(right_idx, div)
            right_step = True

        if left_idx and left_idx != 1 and arr[left_idx - 1] + 1 != left_value:
            check_and_write(left_idx, div)
            left_step = True

        if (not right_step and not left_step) and div < threshold and min_moves > len_moves:
            steps = mask_step(arr)

            if left_idx and left_idx != 1 and left_idx not in steps:
                steps.append(left_idx)
            elif left_idx and left_idx in steps:
                steps.remove(left_idx)

            if right_idx and right_value != 1 and right_idx not in steps:
                steps.append(right_idx)
            elif right_idx and right_idx in steps:
                steps.remove(right_idx)

            for i in steps:
                check_and_write(i, div + 1)

    # ---------------------------------------------------------------------------------

    while not permute_complete:
        move_recursive(tuple(perm), ())
        threshold += 1

    sorted_answer = dict(sorted(permute_complete.items(), key=lambda item: item[1]))

    # return list(sorted_answer.keys()), None, 0, 0
    return sorted_answer

In [ ]:
def pancake_sort_v4(perm, treshold=10):
    """Return a sequence of prefix reversals that sorts `perm` to the identity permutation."""

    n = len(perm)

    arr_max_len = 0
    total_iter = 0
    new_min_stat = 0

    permute_search = {tuple(perm): ((), 0, True)}

    target = tuple(i for i in range(n))

    def check_and_write(idx, _div=0, left_way=True):
        nonlocal new_min_stat
        _stat = stat + _div
        if (not moves or (moves and moves[-1] != idx)) and min_stat + treshold >= _stat:
            arr_ = arr[idx - 1::-1] + arr[idx:]
            old_stat = permute_search.get(arr_)
            if not old_stat:
                permute_search[arr_] = (moves + (idx,), _stat, left_way)
                new_min_stat = min(new_min_stat, _stat)

    def mask_step(perm):
        perm_ = list(perm)
        n = len(perm_)
        step = [] if perm_[n-1] == n-1 else [n]

        while len(perm_) - 2:
            m = perm_.pop()
            if not (perm_[-1] - 1 == m or perm_[-1] + 1 == m):
                step.append(len(perm_))
        return step

    while permute_search and target not in permute_search:

        min_stat = new_min_stat
        new_min_stat = min_stat + n

        for arr in list(permute_search):

            total_iter += 1

            moves, stat, way = permute_search[arr]

            right_step = False
            left_step = False

            # value < arr[0] : left
            left_value = arr[0] - 1 if arr[0] else None
            left_idx = arr.index(left_value) if left_value is not None else None

            # arr[0] < value : right
            right_value = arr[0] + 1 if arr[0] < n - 1 else None
            right_idx = arr.index(right_value) if right_value else n

            if not right_value or (right_idx != 1 and arr[right_idx - 1] - 1 != right_value):
                check_and_write(right_idx, 0.05)
                right_step = True
            elif right_idx != 1:
                check_and_write(right_idx, 2)

            if left_idx and left_idx != 1 and arr[left_idx - 1] + 1 != left_value:
                check_and_write(left_idx, 0.25)
                left_step = True
            elif left_idx and left_idx != 1:
                check_and_write(left_idx, 2)

            if not right_step or not left_step:
                for i in mask_step(arr):
                    check_and_write(i, 2)

            arr_max_len = max(arr_max_len, len(permute_search))
            permute_search.pop(arr)

    return permute_search[target], None, arr_max_len, total_iter

In [ ]:
def pancake_sort_recursive_v3(perm):
    n = len(perm)

    permute_search = {}
    permute_complete = {}

    target = tuple(i for i in range(n))

    # ---------------------------------------------------------------------------------

    def idxs_and_positions(arr):

        # value < arr[0] : left
        left_value = arr[0] - 1 if arr[0] else None
        left_idx = arr.index(left_value) if left_value is not None else None

        # arr[0] < value : right
        right_value = arr[0] + 1 if arr[0] < n - 1 else None
        right_idx = arr.index(right_value) if right_value else n

        return left_value, left_idx, right_value, right_idx


    def mask_step(perm):
        perm_ = list(perm)
        n = len(perm_)
        steps = [] if perm_[n-1] == n-1 else [n]

        _, left_idx, _, right_idx = idxs_and_positions(perm)

        while len(perm_) - 2:
            m = perm_.pop()
            if not (perm_[-1] - 1 == m or perm_[-1] + 1 == m):
                steps.append(len(perm_))

        if left_idx and left_idx not in steps:
            steps.append(left_idx)

        if right_idx and right_idx not in steps:
            steps.append(right_idx)

        return steps


    def move_recursive(arr, moves):

        def check_and_write(idx):
            if not moves or (moves and moves[-1] != idx):
                arr_ = arr[idx - 1::-1] + arr[idx:]
                if arr_ == target:
                    permute_complete[moves + (idx,)] = (target, len(moves) + 1)
                else:
                    move_recursive(arr_, moves + (idx,))


        left_value, left_idx, right_value, right_idx = idxs_and_positions(arr)


        if not right_value or (right_idx != 1 and arr[right_idx - 1] - 1 != right_value):
            check_and_write(right_idx)
        else:
            permute_search[arr] = moves, len(moves)

        if left_idx and left_idx != 1 and arr[left_idx - 1] + 1 != left_value:
            check_and_write(left_idx)
        else:
            permute_search[arr] = moves, len(moves)

    # ---------------------------------------------------------------------------------

    move_recursive(tuple(perm), ())

    if permute_complete:
        return permute_complete, permute_search, None

    # ================

    perm, decode_dict = revers_perm(perm)
    permute_search_forward = permute_search.copy()
    permute_search.clear()

    move_recursive(tuple(perm), ())

    if permute_complete:
        permute_complete_backward = {}
        while permute_complete:
            key, item = permute_complete.popitem()
            permute_complete_backward[key[::-1]] = item
        return permute_complete_backward, permute_search_forward, permute_search

    for perm_backward in permute_search.keys():
        for idx in range(2, n + 1):
            perm = tuple(decode_dict[i] for i in (perm_backward[idx - 1::-1] + perm_backward[idx:]))
            if perm in permute_search_forward:
                steps = permute_search[perm_backward][0] + (idx,)
                steps = permute_search_forward[perm][0] + steps[::-1]
                permute_complete[steps] = len(steps)



    return permute_complete, permute_search_forward, permute_search

## job


In [ ]:
def pancake_sort_recursive_v3_1(perm, percent=0.75):

    n = len(perm)
    perm = tuple(perm)
    max_len_moves = 0

    permute_search = {}
    permute_complete = {}

    target = tuple(i for i in range(n))

    # ---------------------------------------------------------------------------------

    def idxs_and_positions(arr):

        # value < arr[0] : left
        left_value = arr[0] - 1 if arr[0] else None
        left_idx = arr.index(left_value) if left_value is not None else None

        # arr[0] < value : right
        right_value = arr[0] + 1 if arr[0] < n - 1 else None
        right_idx = arr.index(right_value) if right_value else n

        return left_value, left_idx, right_value, right_idx


    def mask_step_including(perm):

        perm_ = list(perm)
        n = len(perm_)
        steps = [] if perm_[n-1] == n-1 else [n]

        _, left_idx, _, right_idx = idxs_and_positions(perm)

        while len(perm_) - 2:
            m = perm_.pop()
            if not (perm_[-1] - 1 == m or perm_[-1] + 1 == m):
                steps.append(len(perm_))

        if left_idx and left_idx not in steps:
            steps.append(left_idx)

        if right_idx and right_idx not in steps:
            steps.append(right_idx)

        return steps

    def check_and_write(arr, moves, idx):
        if not moves or (moves and moves[-1] != idx):
            arr_ = arr[idx - 1::-1] + arr[idx:]
            if arr_ == target:
                permute_complete[moves + (idx,)] = len(moves) + 1
            else:
                move_recursive(arr_, moves + (idx,))


    def move_recursive(arr, moves):
        nonlocal max_len_moves

        if permute_complete:
            return

        len_moves = len(moves)
        max_len_moves = max(len_moves, max_len_moves)

        left_value, left_idx, right_value, right_idx = idxs_and_positions(arr)

        left_step = False
        right_step = False

        if left_idx and left_idx != 1 and arr[left_idx - 1] + 1 != left_value:
            check_and_write(arr, moves, left_idx)
            left_step = True

        if not right_value or (right_idx != 1 and arr[right_idx - 1] - 1 != right_value):
            check_and_write(arr, moves, right_idx)
            right_step = True

        if not left_step and not right_step and len_moves > max_len_moves * percent:
            permute_search[arr] = moves, len_moves

    # ---------------------------------------------------------------------------------

    move_recursive(perm, ())

    if not permute_search:
        permute_search[perm] = (), 0

    while not permute_complete:

        sorted_permute_search = dict(sorted(permute_search.items(), key=lambda item: item[1])[:n**3])

        for perm, (steps, _) in list(sorted_permute_search.items()):
            if permute_complete:
                break
            for idx in mask_step_including(perm):
                check_and_write(perm, steps, idx)
            permute_search.pop(perm)

    # ================

    return list(permute_complete.keys()), permute_search, 0, 0

In [ ]:
def pancake_sort_recursive_v3_2(perm, percent=0.75):

    n = len(perm)
    perm = tuple(perm)
    max_len_moves = 0

    permute_search = {}
    permute_complete = {}

    target = tuple(i for i in range(n))

    # ---------------------------------------------------------------------------------

    def idxs_and_positions(arr):

        # value < arr[0] : left
        left_value = arr[0] - 1 if arr[0] else None
        left_idx = arr.index(left_value) if left_value is not None else None

        # arr[0] < value : right
        right_value = arr[0] + 1 if arr[0] < n - 1 else None
        right_idx = arr.index(right_value) if right_value else n

        return left_value, left_idx, right_value, right_idx


    def mask_step_including(perm):

        perm_ = list(perm)
        n = len(perm_)
        steps = [] if perm_[n-1] == n-1 else [n]

        _, left_idx, _, right_idx = idxs_and_positions(perm)

        while len(perm_) - 2:
            m = perm_.pop()
            if not (perm_[-1] - 1 == m or perm_[-1] + 1 == m):
                steps.append(len(perm_))

        if left_idx and left_idx not in steps:
            steps.append(left_idx)
        elif left_idx and left_idx in steps:
            steps.remove(left_idx)

        if right_idx and right_idx not in steps:
            steps.append(right_idx)
        elif right_idx and right_idx in steps:
            steps.remove(right_idx)

        return steps

    def check_and_write(arr, moves, idx):
        if not moves or (moves and moves[-1] != idx):
            arr_ = arr[idx - 1::-1] + arr[idx:]
            if arr_ == target:
                permute_complete[moves + (idx,)] = len(moves) + 1
            else:
                move_recursive(arr_, moves + (idx,))


    def move_recursive(arr, moves):
        nonlocal max_len_moves

        if permute_complete:
            return

        len_moves = len(moves)
        max_len_moves = max(len_moves, max_len_moves)

        left_value, left_idx, right_value, right_idx = idxs_and_positions(arr)

        left_step = False
        right_step = False

        if left_idx and left_idx != 1 and arr[left_idx - 1] + 1 != left_value:
            check_and_write(arr, moves, left_idx)
            left_step = True

        if not right_value or (right_idx != 1 and arr[right_idx - 1] - 1 != right_value):
            check_and_write(arr, moves, right_idx)
            right_step = True

        if (not left_step or not right_step) and len_moves > max_len_moves * percent:
            permute_search[arr] = moves, len_moves

    # ---------------------------------------------------------------------------------

    move_recursive(perm, ())
    # move_recursive(perm[::-1], (n,))

    if not permute_search:
        permute_search[perm] = (), 0

    while not permute_complete:

        sorted_permute_search = dict(sorted(permute_search.items(), key=lambda item: item[1])[:n**3])

        for perm, (steps, _) in list(sorted_permute_search.items()):
            if permute_complete:
                break
            for idx in mask_step_including(perm):
                check_and_write(perm, steps, idx)
            permute_search.pop(perm)

    # ================

    return list(permute_complete.keys()), permute_search, 0, 0

In [ ]:
import random

In [ ]:
temp_n = [0] * 50
for i in range(50):
    temp_n[i] = 2 * i if i != 0 else 1


In [ ]:

import random
import time

n = 50
# начальное состояние: отсортированная стопка 0..49
b = [0]*n
for i in range(n):
    left = i-1 if i>0 else 0
    right = i+1 if i<n-1 else 0
    b[i] = left + right
first = 0

def find_and_flip(x, first, b):
    # ищем позицию x и сразу получаем параметры для переворота
    prev = 0
    curr = first
    pos = 0
    while True:
        if curr == x:
            # нашли
            p = b[curr] - prev  # правый сосед x
            # выполняем переворот
            b[first] += p
            b[curr] -= p
            if p != 0:
                b[p] += first - curr
            new_first = curr
            return pos, new_first, b
        next_val = b[curr] - prev
        if next_val == 0:
            # дошли до конца, не нашли (ошибка)
            return -1, first, b
        prev, curr = curr, next_val
        pos += 1



In [ ]:
import random
import time

n = 200
# Инициализация: отсортированная стопка 0..49
b = {}
for i in range(n):
    left = i - 1 if i > 0 else -1
    right = i + 1 if i < n - 1 else -1
    b[i] = left + right
first = 0

def find_and_flip(x, first, b):
    prev = -1          # предыдущий элемент (для первого это граница)
    curr = first
    pos = 0
    while True:
        if curr == x:
            # Нашли нужный блин
            p = b[curr] - prev   # следующий за x (правый сосед)
            # Обновляем суммы трёх блинов
            b[first] += p
            b[curr] -= p
            if p != -1:          # если x не последний
                b[p] += first - curr
            new_first = curr
            return pos, new_first, b
        # Вычисляем следующий блин
        next_val = b[curr] - prev
        if next_val == -1:       # дошли до конца, не нашли (ошибка)
            return -1, first, b
        prev, curr = curr, next_val
        pos += 1



In [ ]:
%%time
for _ in range(1_000_000):
    x = random.randint(2, n - 1)
    pos, first, b = find_and_flip(x, first, b)

CPU times: user 487 ms, sys: 0 ns, total: 487 ms
Wall time: 486 ms


In [ ]:
temp = [i for i in range(n)]

In [ ]:
%%time
for _ in range(1000000):
    idx = temp.index(random.randint(2, n - 1)) + 1
    temp[:idx] = temp[idx - 1::-1]

CPU times: user 1.29 s, sys: 0 ns, total: 1.29 s
Wall time: 1.29 s


In [ ]:
temp = tuple(i for i in range(n))

In [ ]:
%%time
for _ in range(1000000):
    idx = temp.index(random.randint(2, n - 1)) + 1
    temp = temp[idx - 1::-1] + temp[idx:]

CPU times: user 1.6 s, sys: 8.65 ms, total: 1.61 s
Wall time: 1.61 s


In [ ]:
def pancake_sort_recursive_v3_3_all_state(perm, percent=0.75):

    n = len(perm)
    perm = tuple(perm)
    max_len_moves = 0

    permute_search = {}
    permute_complete = {}

    target = tuple(i for i in range(n))

    # ---------------------------------------------------------------------------------

    def idxs_and_positions(arr):

        # value < arr[0] : left
        left_value = arr[0] - 1 if arr[0] else None
        left_idx = arr.index(left_value) if left_value is not None else None

        # arr[0] < value : right
        right_value = arr[0] + 1 if arr[0] < n - 1 else None
        right_idx = arr.index(right_value) if right_value else n

        return left_value, left_idx, right_value, right_idx


    def mask_step_including(perm):

        perm_ = list(perm)
        n = len(perm_)
        steps = [] if perm_[n-1] == n-1 else [n]

        _, left_idx, _, right_idx = idxs_and_positions(perm)

        while len(perm_) - 2:
            m = perm_.pop()
            if not (perm_[-1] - 1 == m or perm_[-1] + 1 == m):
                steps.append(len(perm_))

            if left_idx and left_idx not in steps:
                steps.append(left_idx)

            if right_idx and right_idx not in steps:
                steps.append(right_idx)

        return steps


    def check_and_write(arr, moves, idx):

        if not moves or (moves and moves[-1] != idx):
            arr_ = arr[idx - 1::-1] + arr[idx:]
            if arr_ == target:
                permute_complete[moves + (idx,)] = len(moves) + 1
            else:
                move_recursive(arr_, moves + (idx,))


    def move_recursive(arr, moves):

        if permute_complete or permute_search.get(arr):
            return

        len_moves = len(moves)

        permute_search[arr] = moves, len_moves

        left_value, left_idx, right_value, right_idx = idxs_and_positions(arr)

        if left_idx and left_idx != 1 and arr[left_idx - 1] + 1 != left_value:
            check_and_write(arr, moves, left_idx)

        if not right_value or (right_idx != 1 and arr[right_idx - 1] - 1 != right_value):
            check_and_write(arr, moves, right_idx)

    # ---------------------------------------------------------------------------------

    move_recursive(perm, ())

    if not permute_search:
        permute_search[perm] = (), 0

    while not permute_complete:

        for perm, (steps, _) in list(permute_search.items()):
            if permute_complete:
                break
            for idx in mask_step_including(perm):
                check_and_write(perm, steps, idx)
            permute_search.pop(perm)

    # ================

    return list(permute_complete.keys()), permute_search, 0, 0

In [ ]:
def pancake_sort_recursive_v3_4_unlock(perm, percent=0.75):

    n = len(perm)
    perm = tuple(perm)
    max_len_moves = 0

    permute_search = {}
    permute_complete = {}

    target = tuple(i for i in range(n))

    # ---------------------------------------------------------------------------------

    def idxs_and_positions(arr):

        # value < arr[0] : left
        left_value = arr[0] - 1 if arr[0] else None
        left_idx = arr.index(left_value) if left_value is not None else None

        # arr[0] < value : right
        right_value = arr[0] + 1 if arr[0] < n - 1 else None
        right_idx = arr.index(right_value) if right_value else n

        return left_value, left_idx, right_value, right_idx


    def mask_step_including(perm):

        perm_ = list(perm)
        n = len(perm_)
        steps = [] if perm_[n-1] == n-1 else [n]

        _, left_idx, _, right_idx = idxs_and_positions(perm)

        while len(perm_) - 2:
            m = perm_.pop()
            if not (perm_[-1] - 1 == m or perm_[-1] + 1 == m):
                steps.append(len(perm_))

            if left_idx and left_idx not in steps:
                steps.append(left_idx)

            if right_idx and right_idx not in steps:
                steps.append(right_idx)

        return steps


    def check_and_write(arr, moves, idx):

        if not moves or (moves and moves[-1] != idx):
            arr_ = arr[idx - 1::-1] + arr[idx:]
            if arr_ == target:
                permute_complete[moves + (idx,)] = len(moves) + 1
            else:
                move_recursive(arr_, moves + (idx,))


    def move_recursive(arr, moves):

        if permute_complete:
            return

        len_moves = len(moves)

        left_value, left_idx, right_value, right_idx = idxs_and_positions(arr)

        if left_idx and left_idx != 1 and arr[left_idx - 1] + 1 != left_value:
            check_and_write(arr, moves, left_idx)

        if not right_value or (right_idx != 1 and arr[right_idx - 1] - 1 != right_value):
            check_and_write(arr, moves, right_idx)

    # ---------------------------------------------------------------------------------

    move_recursive(perm, ())

    if permute_complete:
        return list(permute_complete.keys()), None, 0, 0
    else:
        return [(0,) * (n * 2)], None, 0, 0

In [ ]:
perm = tuple(map(lambda x: int(x) - 1, '1 5 2 7 4 6 3 8 14 11 13 10 12 9 15 20 17 19 16 18'.split()))

In [ ]:
idx = 655

perm = parse_permutation(test_df.iloc[idx].permutation)
best_df.iloc[idx]

,655
permutation,"10,19,15,18,16,2,9,17,14,12,0,6,7,11,1,13,3,8,5,4"
solution,R13.R2.R18.R11.R9.R13.R6.R5.R9.R20.R13.R9.R4.R...
score,19
prob_step,18
n,20


In [ ]:
perm = tuple(map(int, '0,6,4,2,5,3,1,7,13,11,9,12,10,8,14,20,18,16,19,17,15,21,27,25,23,26,24,22'.split(',')))

In [ ]:
perm, decode_dict = revers_perm(perm)
perm

[10, 14, 5, 16, 19, 18, 11, 12, 17, 6, 0, 13, 9, 15, 8, 2, 4, 7, 3, 1]

In [ ]:
perm_decode = perm.copy()

for i in range(best_df.n.iloc[idx]):
    perm_decode = [decode_dict[j] for j in perm_decode]
    perm_code = pancake_sort_recursive_v3_2(perm_decode, percent=0)[0][0]

    print(f'{i} : {perm_decode} : {len(perm_code)} : {perm_code}')

0 : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19] : 1 : (1,)
1 : [10, 19, 15, 18, 16, 2, 9, 17, 14, 12, 0, 6, 7, 11, 1, 13, 3, 8, 5, 4] : 19 : (6, 14, 12, 2, 6, 12, 9, 20, 3, 9, 18, 16, 3, 14, 13, 11, 6, 4, 12)
2 : [0, 4, 13, 5, 3, 15, 12, 8, 1, 7, 10, 9, 17, 6, 19, 11, 18, 14, 2, 16] : 20 : (8, 20, 7, 10, 7, 8, 18, 13, 6, 19, 9, 10, 14, 5, 17, 20, 12, 18, 15, 4)
3 : [10, 16, 11, 2, 18, 13, 7, 14, 19, 17, 0, 12, 8, 9, 4, 6, 5, 1, 15, 3] : 19 : (2, 18, 14, 5, 4, 7, 13, 2, 7, 17, 13, 19, 14, 20, 14, 4, 9, 3, 17)
4 : [0, 3, 6, 15, 5, 11, 17, 1, 4, 8, 10, 7, 14, 12, 16, 9, 2, 19, 13, 18] : 21 : (7, 14, 12, 2, 18, 20, 15, 4, 10, 3, 18, 13, 12, 4, 9, 4, 14, 11, 5, 19, 3)
5 : [10, 18, 9, 13, 2, 6, 8, 19, 16, 14, 0, 17, 1, 7, 3, 12, 15, 4, 11, 5] : 21 : (2, 7, 4, 9, 16, 18, 16, 8, 7, 5, 17, 11, 19, 14, 8, 9, 3, 20, 10, 15, 5)
6 : [0, 5, 12, 11, 15, 9, 14, 4, 3, 1, 10, 8, 19, 17, 18, 7, 13, 16, 6, 2] : 18 : (9, 19, 3, 12, 17, 2, 13, 4, 2, 6, 16, 20, 18, 4, 7, 5, 11, 20)

In [ ]:
%%time
permute_complete_, permute_search_, _, _ = pancake_sort_recursive_v3_2(perm, percent=0.75)

CPU times: user 382 µs, sys: 29 µs, total: 411 µs
Wall time: 415 µs


In [ ]:
print(len(permute_complete_[0]), ' : ', permute_complete_, ' : ', best_df.solution.iloc[idx])

19  :  [(12, 4, 6, 11, 13, 14, 3, 7, 3, 20, 4, 2, 12, 14, 3, 7, 2, 13, 14)]  :  R13.R2.R18.R11.R9.R13.R6.R5.R9.R20.R13.R9.R4.R2.R18.R19.R16.R10.R12


In [ ]:
# del permute_search_forward
del permute_search
gc.collect()

100

In [ ]:
%%time
# perm = parse_permutation(test_df.iloc[idx].permutation)

r, permute, p, i = pancake_sort_v4(perm, 5)
print(perm)
print(f'Len: {len(perm)} | Steps: {len(r[0])} | max line: {p} | iter: {i}')
print(r, '\n')


(0, 6, 4, 2, 5, 3, 1, 7, 13, 11, 9, 12, 10, 8, 14, 20, 18, 16, 19, 17, 15, 21, 27, 25, 23, 26, 24, 22)
Len: 28 | Steps: 32 | max line: 35373003 | iter: 181291920
((6, 3, 2, 4, 3, 28, 3, 5, 4, 3, 2, 6, 28, 21, 3, 5, 4, 3, 2, 6, 21, 14, 3, 5, 4, 3, 2, 6, 14, 5, 7, 2), 9.999999999999998, True) 

CPU times: user 23min 5s, sys: 26.3 s, total: 23min 32s
Wall time: 24min 12s


In [ ]:
%%time
permute_complete, permute_search_forward, permute_search = pancake_sort_recursive_v3(perm)

CPU times: user 359 µs, sys: 0 ns, total: 359 µs
Wall time: 932 µs


In [ ]:
%%time
permute_complete_, permute_search_, _, _ = pancake_sort_recursive_v3_2(perm, percent=0)

CPU times: user 3.27 ms, sys: 201 µs, total: 3.47 ms
Wall time: 3.47 ms


In [ ]:
%%time
permute_complete_all, permute_search_all, _, _ = pancake_sort_recursive_v3_3_all_state(perm, percent=0)

CPU times: user 3.05 ms, sys: 0 ns, total: 3.05 ms
Wall time: 3.06 ms


In [ ]:
%%time
permute_complete_2 = pancake_sort_recursive_v2_1(perm, 1)

CPU times: user 3.56 ms, sys: 217 µs, total: 3.78 ms
Wall time: 3.78 ms


In [ ]:
print_search(permute_complete)

None


In [ ]:
permute_complete_

[(11, 8, 2, 9, 3, 20, 15, 4, 12, 14, 9, 12, 8, 16, 15, 19, 10, 3, 14)]

In [ ]:
len(permute_complete_all)
permute_complete_all

[(16, 8, 13, 11, 14, 13, 15, 8, 4, 20, 3, 17, 6, 8, 11, 15, 5, 14)]

In [ ]:
print_search(permute_complete_2)

(14, 3, 10, 8, 12, 8, 7, 4, 19, 14, 10, 17, 11, 3, 6, 12, 20, 10, 19) | 19
(14, 3, 10, 8, 12, 8, 17, 14, 19, 4, 13, 20, 3, 14, 19, 17, 7, 3, 9) | 19
(14, 3, 10, 8, 12, 8, 17, 12, 19, 4, 15, 6, 8, 3, 20, 10, 7, 3, 9) | 19
(14, 3, 10, 8, 12, 8, 17, 8, 6, 12, 19, 4, 15, 3, 20, 10, 7, 3, 9) | 19
(14, 3, 10, 8, 12, 8, 17, 2, 7, 5, 12, 19, 4, 15, 20, 10, 7, 3, 9) | 19
(14, 3, 10, 8, 12, 7, 19, 17, 19, 12, 6, 17, 20, 14, 18, 2, 15, 3, 20) | 19
(14, 3, 10, 8, 12, 7, 8, 17, 10, 19, 8, 2, 11, 16, 14, 5, 17, 20, 10) | 19
(14, 3, 10, 19, 17, 2, 9, 6, 20, 12, 7, 10, 14, 9, 17, 14, 19, 17, 11) | 19
(14, 3, 10, 19, 17, 2, 15, 5, 9, 6, 11, 9, 12, 20, 6, 4, 9, 5, 11) | 19
(14, 3, 10, 19, 17, 2, 15, 5, 9, 6, 11, 9, 14, 18, 5, 16, 20, 2, 11) | 19
(14, 12, 6, 5, 9, 11, 3, 4, 19, 14, 10, 17, 11, 3, 6, 12, 20, 10, 19) | 19
(14, 12, 5, 6, 17, 8, 14, 19, 17, 6, 3, 11, 18, 5, 11, 20, 7, 4, 16) | 19


In [ ]:
print_search(permute_search_all)

(5, 6, 3, 4, 1, 2, 11, 8, 7, 0, 9, 10) | ((3, 12), 2)
(3, 4, 1, 2, 11, 8, 7, 0, 9, 10, 6, 5) | ((3, 10), 2)
(3, 4, 0, 1, 2, 11, 10, 9, 8, 7, 6, 5) | ((3, 5, 3, 8, 1, 7, 10), 7)
(7, 8, 9, 10, 11, 2, 1, 0, 4, 3, 6, 5) | ((3, 5, 3, 8, 1, 7), 6)
(5, 6, 3, 4, 1, 2, 0, 7, 8, 9, 10, 11) | ((6, 3, 5, 3, 12), 5)
(11, 10, 9, 8, 7, 0, 2, 1, 4, 3, 6, 5) | ((6, 3, 5, 3), 4)
(5, 6, 3, 4, 1, 2, 11, 0, 9, 10, 7, 8) | ((5, 12), 2)
(3, 4, 1, 2, 11, 0, 9, 10, 7, 8, 6, 5) | ((5, 10), 2)
(5, 6, 3, 4, 8, 7, 10, 9, 0, 1, 2, 11) | ((5, 8, 3, 12), 4)
(11, 2, 1, 0, 9, 10, 7, 8, 4, 3, 6, 5) | ((5, 8, 3), 3)
(1, 2, 11, 0, 9, 10, 7, 8, 4, 3, 6, 5) | ((5, 8), 2)
(5, 6, 3, 4, 1, 2, 8, 7, 10, 9, 0, 11) | ((5, 6, 12), 3)
(11, 0, 9, 10, 7, 8, 2, 1, 4, 3, 6, 5) | ((5, 6), 2)
(9, 10, 7, 8, 0, 11, 2, 1, 4, 3, 6, 5) | ((5, 4), 2)
(3, 4, 1, 2, 11, 0, 9, 10, 8, 7, 6, 5) | ((5, 2, 10), 3)
(7, 8, 10, 9, 0, 11, 2, 1, 4, 3, 6, 5) | ((5, 2), 2)
(0, 9, 8, 7, 10, 11, 2, 1, 4, 3, 6, 5) | ((5, 3, 5), 3)
(10, 7, 8, 9, 0, 11, 2, 1, 4, 

In [ ]:
print_search(permute_search_)

(0, 1, 2, 8, 9, 10, 16, 15, 14, 13, 12, 11, 3, 5, 4, 19, 6, 7, 17, 18) | ((11, 8, 20, 15, 17, 7, 10, 9, 3, 8), 10)
(15, 16, 10, 9, 8, 2, 1, 0, 14, 13, 12, 11, 3, 5, 4, 19, 6, 7, 17, 18) | ((11, 8, 20, 15, 17, 7, 10, 9, 3), 9)
(10, 16, 15, 9, 8, 2, 1, 0, 14, 13, 12, 11, 3, 5, 4, 19, 6, 7, 17, 18) | ((11, 8, 20, 15, 17, 7, 10, 9), 8)
(7, 6, 19, 4, 5, 3, 11, 12, 13, 10, 9, 8, 2, 1, 0, 14, 15, 16, 17, 18) | ((11, 8, 20, 15, 17, 7, 10, 6, 8, 18), 10)
(16, 15, 14, 0, 1, 2, 8, 9, 10, 13, 12, 11, 3, 5, 4, 19, 6, 7, 17, 18) | ((11, 8, 20, 15, 17, 7, 10, 6, 8), 9)
(9, 8, 2, 1, 0, 14, 15, 16, 10, 13, 12, 11, 3, 5, 4, 19, 6, 7, 17, 18) | ((11, 8, 20, 15, 17, 7, 10, 6), 8)
(0, 1, 2, 8, 9, 15, 16, 10, 13, 14, 12, 11, 3, 5, 4, 19, 6, 7, 17, 18) | ((11, 8, 20, 15, 17, 7, 9), 7)
(8, 9, 15, 16, 10, 11, 12, 13, 14, 0, 1, 2, 3, 5, 4, 19, 6, 7, 17, 18) | ((11, 8, 20, 15, 17, 12, 6, 4, 3, 11), 10)
(1, 0, 14, 13, 12, 11, 10, 16, 15, 9, 8, 2, 3, 5, 4, 19, 6, 7, 17, 18) | ((11, 8, 20, 15, 17, 12, 6, 4, 3), 9)


In [ ]:
print_search(permute_search_forward)

(5, 4, 19, 18, 17, 7, 6, 8, 2, 3, 11, 12, 14, 0, 1, 13, 10, 16, 15, 9) | ((11, 8, 6, 3, 9), 5)
(2, 8, 6, 7, 17, 18, 19, 4, 5, 3, 11, 12, 14, 0, 1, 13, 10, 16, 15, 9) | ((11, 8, 6, 3), 4)
(6, 8, 2, 7, 17, 18, 19, 4, 5, 3, 11, 12, 14, 0, 1, 13, 10, 16, 15, 9) | ((11, 8, 6), 3)
(18, 17, 7, 2, 8, 6, 19, 4, 5, 3, 11, 12, 14, 0, 1, 13, 10, 16, 15, 9) | ((11, 8), 2)
(15, 16, 10, 13, 1, 0, 14, 12, 11, 3, 4, 19, 6, 5, 18, 17, 7, 2, 8, 9) | ((11, 9, 6, 19), 4)
(0, 1, 13, 10, 16, 15, 14, 12, 11, 3, 4, 19, 6, 5, 18, 17, 7, 2, 8, 9) | ((11, 9, 6, 19, 6), 5)
(2, 8, 7, 17, 18, 5, 6, 19, 4, 3, 11, 12, 14, 0, 1, 13, 10, 16, 15, 9) | ((11, 9, 6, 2), 4)
(16, 15, 9, 10, 13, 1, 0, 14, 12, 11, 3, 4, 5, 18, 17, 7, 2, 8, 6, 19) | ((11, 9, 8, 20, 3), 5)
(2, 7, 17, 18, 5, 4, 3, 11, 12, 14, 0, 1, 13, 10, 16, 15, 9, 8, 6, 19) | ((11, 9, 8, 20, 17), 5)
(19, 6, 8, 2, 7, 17, 18, 5, 4, 3, 11, 12, 14, 0, 1, 13, 10, 16, 15, 9) | ((11, 9, 8), 3)
(0, 1, 13, 14, 12, 4, 19, 6, 8, 2, 7, 17, 18, 5, 3, 11, 10, 16, 15, 9) | ((

In [ ]:
print_search(permute_search)

(10, 11, 8, 9, 6, 7, 0, 1, 2, 3, 4, 5) | ((9, 11, 9), 3)
(2, 1, 0, 7, 6, 9, 8, 11, 10, 3, 4, 5) | ((9, 11), 2)
(4, 3, 10, 11, 8, 9, 6, 7, 0, 1, 2, 5) | ((9,), 1)
(0, 7, 6, 9, 8, 11, 10, 3, 4, 1, 2, 5) | ((), 0)


In [ ]:
test_df_ = best_df.loc[best_df.score - 1 > best_df.prob_step]

In [ ]:
test_df_

,id,permutation,solution,score,prob_step,n
12,12,"1,8,6,3,4,11,2,9,10,0,7,5",R9.R2.R7.R11.R10.R3.R12.R3.R5.R8.R11.R5,12,10,12
15,15,"10,11,1,2,8,9,4,3,5,7,0,6",R10.R11.R3.R12.R2.R5.R2.R7.R2.R10,10,8,12
20,20,"0,9,10,7,8,11,2,1,4,3,6,5",R6.R3.R10.R12.R10.R2.R6.R12.R6.R9,10,7,12
28,28,"9,0,2,3,11,7,8,5,6,1,10,4",R12.R8.R12.R9.R4.R8.R11.R5.R10.R9.R11,11,9,12
29,29,"10,3,5,7,6,4,9,8,2,11,0,1",R6.R4.R3.R2.R5.R8.R3.R9.R10.R12.R2,11,9,12
...,...,...,...,...,...,...
1231,1231,"26,34,5,31,22,13,29,21,2,18,9,24,15,6,14,30,1,...",R34.R27.R29.R21.R30.R25.R23.R11.R33.R18.R24.R1...,34,32,35
1586,1586,"5,23,18,27,26,2,17,7,30,37,21,20,24,16,33,34,1...",R35.R40.R35.R31.R29.R39.R18.R4.R9.R6.R24.R21.R...,36,34,40
1949,1949,"42,31,44,37,22,34,9,30,26,18,32,41,48,43,8,7,1...",R11.R41.R22.R38.R6.R10.R28.R21.R43.R11.R23.R21...,47,45,50
2250,2250,"72,10,14,78,51,56,85,75,77,31,71,73,37,65,90,5...",R11.R84.R64.R82.R5.R13.R56.R24.R19.R81.R80.R53...,102,100,100


In [ ]:
test_df_ = test_df[405:605] # 16

In [ ]:
test_df_ = test_df[605:805] # 20

In [ ]:
test_df_ = test_df[805:1005] # 25

In [ ]:
test_df_ = test_df[1005:1205] # 30 - 12

In [ ]:
test_df_ = test_df[1205:1405] # 35 - 12

In [ ]:
test_df_ = test_df[1405:1605] # 40 - 12

In [ ]:
test_df_ = test_df[1605:1805] # 45 - 12

In [ ]:
test_df_ = test_df[1805:2005] # 50 - 8

In [ ]:
test_df_ = test_df[2005:2205] # 75 - 6

In [ ]:
test_df_ = test_df[2205:] # 100 - 5

In [ ]:
test_df_ = test_df[2268:]

In [ ]:
test_df_ = test_df[:1005]

In [ ]:
test_df_ = test_df.copy()

In [ ]:
%%time

result = []

for row in tqdm(test_df_.to_dict('records'), total=len(test_df_), position=0):
    result.append(process_row(row, pancake_sort_recursive_v2_1, treshold=1, save=False, from_target=False))

submission_df = pd.DataFrame(result).sort_values('id').reset_index(drop=True)

print(
    f'Score: {submission_df.score.sum()} | '
    f'n: {submission_df.n.sum()} | '
    f'max len: {max(submission_df.mlen)} | '
    f'sum max len : {submission_df.mlen.sum()} | '
    f'total iter: {submission_df.iter.sum()}'
)

submission_df.head()

  0%|          | 0/1005 [00:00<?, ?it/s]

Score: 16425 | n: 17625 | max len: 0 | sum max len : 0 | total iter: 0
CPU times: user 14.5 s, sys: 51.1 ms, total: 14.6 s
Wall time: 14.5 s


,id,permutation,solution,score,n,mlen,iter
0,0,"3,2,0,1,4",R4.R2,2,5,0,0
1,1,"4,2,3,1,0",R5.R4.R2.R4,4,5,0,0
2,2,"3,1,2,0,4",R4.R3.R2.R3,4,5,0,0
3,3,"1,2,0,3,4",R2.R3,2,5,0,0
4,4,"3,0,4,2,1",R3.R5.R3.R4,4,5,0,0


In [ ]:
# Score: 16451 | n: 17625 | max len: 6863 | sum max len : 610487 | total iter: 4619036
# CPU times: user 19.7 s, sys: 70 ms, total: 19.8 s
# Wall time: 19.7 s

In [ ]:
submission_df = pd.DataFrame(result).sort_values('id').reset_index(drop=True)

In [ ]:
submission_df.loc[submission_df.score != submission_df_.score]

,id,permutation,solution,score,n,mlen,iter
38,243,"5,4,10,13,3,11,7,6,14,12,8,2,0,1,9",R7.R10.R8.R11.R14.R2.R15.R9.R11.R7.R6.R4.R8.R15,14,15,54,420


In [ ]:
submission_df_.loc[submission_df.score != submission_df_.score]

,id,permutation,solution,score,n,mlen,iter
38,243,"5,4,10,13,3,11,7,6,14,12,8,2,0,1,9",R15.R4.R10.R4.R15.R2.R4.R10.R12.R3.R14.R4.R10....,15,15,28,201


In [ ]:
# Score: 16439 | n: 17625 | max len: 108282 | sum max len : 4829590 | total iter: 34596637
# CPU times: user 2min 44s, sys: 380 ms, total: 2min 44s
# Wall time: 2min 44s

In [ ]:
test_df_ = best_df.loc[(best_df.n == 100) & (best_df.score - 3 > best_df.prob_step)]

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed

rows = test_df_.to_dict('records')
results = []

with ProcessPoolExecutor(max_workers=15) as executor:
    futures = {executor.submit(process_row, perm, pancake_sort_recursive_v3_4_unlock,
                               treshold=0.0, save=False, from_target=False) for perm in rows}

    for future in tqdm(as_completed(futures), total=len(futures)):
        results.append(future.result())


  0%|          | 0/120 [00:00<?, ?it/s]

In [ ]:
submission_df = pd.DataFrame(results).sort_values('id').reset_index(drop=True)

print(
    f'Score: {submission_df.score.sum()} | '
    f'n: {submission_df.n.sum()} | '
    f'max len: {max(submission_df.mlen)} | '
    f'sum max len : {submission_df.mlen.sum()} | '
    f'total iter: {submission_df.iter.sum()}'
)

submission_df.head()


Score: 23900 | n: 12000 | max len: 0 | sum max len : 0 | total iter: 0


,id,permutation,solution,score,n,mlen,iter
0,2205,"30,57,38,33,51,77,8,55,90,82,96,91,63,9,92,67,...",R0.R0.R0.R0.R0.R0.R0.R0.R0.R0.R0.R0.R0.R0.R0.R...,200,100,0,0
1,2207,"61,75,30,2,77,12,98,32,67,53,13,51,10,9,72,38,...",R0.R0.R0.R0.R0.R0.R0.R0.R0.R0.R0.R0.R0.R0.R0.R...,200,100,0,0
2,2208,"92,39,45,4,28,30,58,91,22,10,41,84,43,40,23,57...",R0.R0.R0.R0.R0.R0.R0.R0.R0.R0.R0.R0.R0.R0.R0.R...,200,100,0,0
3,2209,"90,87,34,30,89,43,99,9,23,6,95,52,78,7,13,27,6...",R0.R0.R0.R0.R0.R0.R0.R0.R0.R0.R0.R0.R0.R0.R0.R...,200,100,0,0
4,2212,"16,41,64,34,20,25,14,6,61,76,44,97,96,74,89,91...",R0.R0.R0.R0.R0.R0.R0.R0.R0.R0.R0.R0.R0.R0.R0.R...,200,100,0,0


In [ ]:
best_df, result = best_solution(submission_df, safe=True)
print(*result.items())

Best submission updated.
('best', 1) ('same', 0) ('worse', 119)


In [ ]:
compare()

n: 5 | sum n: 25 | score: 16 | prob step: 14 | potential: 2
n: 12 | sum n: 2400 | score: 2166 | prob step: 2038 | potential: 128
n: 15 | sum n: 3000 | score: 2747 | prob step: 2611 | potential: 136
n: 16 | sum n: 3200 | score: 2961 | prob step: 2813 | potential: 148
n: 20 | sum n: 4000 | score: 3773 | prob step: 3636 | potential: 137
n: 25 | sum n: 5000 | score: 4756 | prob step: 4625 | potential: 131
n: 30 | sum n: 6000 | score: 5757 | prob step: 5622 | potential: 135
n: 35 | sum n: 7000 | score: 6732 | prob step: 6591 | potential: 141
n: 40 | sum n: 8000 | score: 7729 | prob step: 7594 | potential: 135
n: 45 | sum n: 9000 | score: 8718 | prob step: 8587 | potential: 131
n: 50 | sum n: 10000 | score: 9748 | prob step: 9618 | potential: 130
n: 75 | sum n: 15000 | score: 14728 | prob step: 14614 | potential: 114
n: 100 | sum n: 20000 | score: 19744 | prob step: 19623 | potential: 121

sum n: 92625 | score: 89575 | prob step: 87986


In [ ]:
best_df.to_csv(BEST_SUBMISSION_PATH, index=False)

In [ ]:
best_df[:1005].loc[best_df[:1005].score != submission_df.score]

,id,permutation,solution,score,prob_step,n
90,90,"11,5,3,4,1,2,6,0,9,8,7,10",R12.R11.R5.R10.R4.R8.R2.R7.R10.R3,10,8,12
217,217,"4,11,12,8,14,9,5,1,10,0,3,13,2,7,6",R10.R15.R11.R2.R7.R8.R4.R9.R12.R9.R13.R14.R5.R15,14,13,15
418,418,"9,6,2,13,12,4,0,5,11,10,3,1,7,8,15,14",R8.R6.R10.R2.R7.R16.R7.R2.R15.R12.R9.R7.R16.R5,14,12,16
476,476,"4,1,13,2,0,7,8,10,6,9,12,11,3,5,15,14",R12.R4.R16.R6.R8.R5.R15.R4.R3.R11.R3.R14.R16.R7,14,13,16
879,879,"18,19,14,7,9,24,1,21,16,22,8,17,12,13,11,20,3,...",R11.R7.R20.R12.R10.R17.R2.R6.R23.R4.R20.R10.R5...,23,22,25


In [ ]:
submission_df.loc[best_df[:1005].score != submission_df.score]

,id,permutation,solution,score,n,mlen,iter
90,90,"11,5,3,4,1,2,6,0,9,8,7,10",R11.R4.R7.R6.R7.R12.R5.R2.R9.R6.R12,11,12,64,228
217,217,"4,11,12,8,14,9,5,1,10,0,3,13,2,7,6",R7.R14.R10.R7.R4.R2.R5.R13.R2.R5.R9.R5.R15.R7.R3,15,15,215,1319
418,418,"9,6,2,13,12,4,0,5,11,10,3,1,7,8,15,14",R16.R12.R16.R6.R16.R4.R11.R5.R8.R13.R6.R5.R7.R...,15,16,107,632
476,476,"4,1,13,2,0,7,8,10,6,9,12,11,3,5,15,14",R13.R3.R10.R7.R3.R8.R12.R10.R9.R15.R16.R15.R9....,15,16,264,1478
879,879,"18,19,14,7,9,24,1,21,16,22,8,17,12,13,11,20,3,...",R11.R6.R25.R8.R23.R13.R8.R20.R12.R11.R15.R2.R1...,24,25,2521,16766


In [ ]:
best_df[:1005].loc[best_df[:1005].score != submission_df.score]

,id,permutation,solution,score,prob_step,n
235,235,"8,12,14,10,7,2,0,11,1,4,3,6,5,9,13",R11.R5.R2.R9.R15.R5.R12.R3.R9.R13.R2.R12.R14.R11,14,13,15
507,507,"2,1,6,5,13,12,11,3,0,4,8,7,15,14,10,9",R7.R14.R2.R16.R5.R14.R7.R3.R4.R9.R4.R9,12,9,16
588,588,"4,15,2,12,3,7,6,9,1,11,13,0,10,8,14,5",R16.R15.R16.R12.R8.R3.R9.R3.R10.R15.R2.R13.R3....,16,15,16
844,844,"24,21,7,13,20,19,14,1,3,4,16,22,17,11,12,6,2,9...",R25.R13.R19.R21.R23.R17.R13.R12.R2.R16.R18.R6....,22,21,25
962,962,"16,9,8,13,1,3,0,24,10,17,15,21,7,14,20,4,6,2,1...",R10.R9.R7.R21.R23.R13.R15.R9.R20.R18.R15.R9.R1...,23,22,25
973,973,"8,4,5,17,6,23,14,24,22,7,3,2,11,1,20,21,0,9,18...",R17.R22.R8.R15.R5.R3.R25.R23.R24.R17.R10.R2.R1...,21,19,25


In [ ]:
submission_df.loc[best_df[:1005].score != submission_df.score]

,id,permutation,solution,score,n,mlen,iter
235,235,"8,12,14,10,7,2,0,11,1,4,3,6,5,9,13",R13.R3.R7.R2.R11.R15.R14.R9.R11.R13.R10.R8.R7....,16,15,360,1425
507,507,"2,1,6,5,13,12,11,3,0,4,8,7,15,14,10,9",R7.R14.R11.R9.R7.R3.R4.R16.R7.R16.R9.R4.R9,13,16,59,229
588,588,"4,15,2,12,3,7,6,9,1,11,13,0,10,8,14,5",R15.R13.R2.R8.R7.R5.R10.R11.R7.R10.R12.R7.R14....,17,16,589,3472
844,844,"24,21,7,13,20,19,14,1,3,4,16,22,17,11,12,6,2,9...",R25.R24.R10.R19.R14.R21.R3.R14.R6.R17.R19.R9.R...,23,25,659,5211
962,962,"16,9,8,13,1,3,0,24,10,17,15,21,7,14,20,4,6,2,1...",R9.R24.R7.R19.R8.R11.R20.R21.R19.R14.R6.R9.R17...,24,25,1317,11395
973,973,"8,4,5,17,6,23,14,24,22,7,3,2,11,1,20,21,0,9,18...",R17.R3.R5.R23.R5.R9.R10.R20.R5.R9.R3.R25.R23.R...,22,25,4333,27221


In [ ]:
best_df = best_df.reindex(columns=['id', 'permutation', 'solution', 'score', 'prob_step', 'n'])

In [ ]:
check_steps(best_df)

In [ ]:
# best_df = best_df.set_index('id').join(test_df.set_index('id')[['n']], how='left').reset_index()
# best_sub_df = submission_df.reindex(columns=['id', 'permutation', 'solution', 'score', 'prob_step'])
# best_df.prob_step = best_df.permutation.apply(lambda x: prob_step(parse_permutation(x)))
# best_sub_df.to_csv(BEST_SUBMISSION_PATH)


In [ ]:
best_df

,id,permutation,solution,score,prob_step,n
0,0,"3,2,0,1,4",R4.R2,2,2,5
1,1,"4,2,3,1,0",R5.R4.R2.R4,4,3,5
2,2,"3,1,2,0,4",R4.R2.R3.R2,4,3,5
3,3,"1,2,0,3,4",R2.R3,2,2,5
4,4,"3,0,4,2,1",R3.R5.R3.R4,4,4,5
...,...,...,...,...,...,...
2400,2400,"47,85,55,96,30,63,54,34,80,69,50,78,14,36,99,6...",R70.R36.R27.R12.R7.R80.R99.R90.R23.R46.R98.R28...,99,98,100
2401,2401,"14,95,85,15,37,38,82,62,86,53,29,17,23,36,43,4...",R3.R8.R18.R39.R31.R40.R96.R12.R79.R74.R59.R70....,99,98,100
2402,2402,"2,74,72,89,0,77,45,15,93,96,1,99,9,50,17,7,65,...",R44.R21.R92.R27.R9.R12.R97.R4.R29.R30.R25.R15....,99,99,100
2403,2403,"38,99,25,79,80,26,27,76,85,18,55,87,62,15,23,8...",R95.R75.R23.R16.R39.R65.R68.R69.R37.R63.R50.R6...,98,98,100


In [ ]:
best_df.prob_step = best_df.permutation.apply(lambda x: prob_step(parse_permutation(x)))

In [ ]:
out_submission_df = best_df[['id', 'permutation', 'solution']]

In [ ]:
out_submission_df.sample(5, random_state=0)

,id,permutation,solution
2010,2010,"43,53,36,54,39,29,52,61,70,33,71,6,8,64,26,38,...",R51.R50.R5.R60.R36.R52.R62.R64.R70.R14.R11.R53...
2286,2286,"1,77,94,0,68,67,12,95,62,99,15,42,57,31,34,86,...",R26.R73.R42.R87.R76.R75.R77.R50.R90.R6.R4.R50....
2100,2100,"30,45,2,15,28,47,61,59,17,4,60,51,5,16,73,38,3...",R22.R35.R58.R53.R38.R63.R70.R58.R45.R15.R54.R5...
2090,2090,"67,40,71,65,60,58,0,11,69,17,30,25,48,13,59,15...",R56.R10.R44.R18.R43.R14.R29.R16.R59.R75.R50.R5...
1070,1070,"7,16,25,29,6,18,11,3,24,13,0,2,9,1,26,4,10,21,...",R4.R30.R24.R16.R29.R21.R3.R6.R20.R6.R9.R28.R5....


In [ ]:
out_submission_df.to_csv(SUBMISSION_PATH, index=False)

In [ ]:
SUBMISSION_PATH

PosixPath('/content/Pancake/submission.csv')

In [ ]:
submission_2point = pd.read_csv('/content/Pancake/submission4.csv')

In [ ]:
submission_2point['score'] = submission_2point.solution.apply(lambda x: len(x.split('.')))

In [ ]:
points_df = submission_2point.loc[best_df.score > submission_2point.score]